# Fuzzy search in historical Dutch text

This notebook demonstrates `fuzzy-search` on a classic use case: resolutions
of the meeting of the Dutch States General (early 18th century), as
digitized via OCR/HTR. The text contains historical spelling, abbreviations
in Latin, and OCR errors (e.g. `&` for `Æ`, broken characters), all of which
make exact string matching unreliable.

We will:
1. Search for a handful of recurring formulaic phrases and dates.
2. Use phrase **variants** and **distractors** to disambiguate similar phrases.
3. Build a **template** that requires several phrases to co-occur in a
   specific order, to detect a higher-level structure in the text.


In [1]:
from fuzzy_search import FuzzyPhraseSearcher


## 1. Basic fuzzy phrase search

Resolutions typically open with the weekday, the date, and the names of the
chair (`PRAESIDE`) and attendants (`PRAESENTIBUS`) of the meeting. These are
formulaic Latin/Dutch phrases that recur across thousands of resolutions, but
rarely transcribe perfectly because of OCR/HTR noise.

We register a few of these phrases, including a date pattern with `.` as a
wildcard character (any character matches), and search a noisy OCR'd text.


In [2]:
config = {
    "char_match_threshold": 0.6,
    "ngram_threshold": 0.5,
    "levenshtein_threshold": 0.6,
    "ignorecase": False,
    "max_length_variance": 3,
    "ngram_size": 2,
    "skip_size": 2,
}

domain_phrases = [
    "PRAESIDE",
    "PRAESENTIBUS",
    "Veneris",       # Friday, in Latin
    "Mercurii",      # Wednesday, in Latin
    "den .. Januarii 1725",  # any date in January 1725
]

fuzzy_searcher = FuzzyPhraseSearcher(config=config, phrase_model=domain_phrases)


In [3]:
text = ("ie Veucris den 5. Januaris 1725. PR&ASIDE, Den Heere Bentinck. PRASENTIEBUS, "
        "De Heeren Jan Welderen , van Dam, Torck , met een extraordinaris Gedeputeerde "
        "uyt de Provincie van Gelderlandt.")

matches = fuzzy_searcher.find_matches(text)
for match in matches:
    print(f"{match.phrase.phrase_string:<22}{match.string:<22}offset={match.offset:<4}"
          f"sim={match.levenshtein_similarity:.2f}")


Veneris               Veucris               offset=3   sim=0.71
den .. Januarii 1725  den 5. Januaris 1725  offset=11  sim=0.90
PRAESIDE              PR&ASIDE              offset=33  sim=0.88
PRAESENTIBUS          PRASENTIEBUS          offset=63  sim=0.92


Despite OCR damage (`Veucris` for `Veneris`, `PR&ASIDE` for `PRAESIDE`,
`PRASENTIEBUS` for `PRAESENTIBUS`, and a digit/letter mix-up in the date),
all four phrases are found with reasonable similarity scores.

## 2. Searching many texts, with text identifiers

When processing a corpus, you typically pass texts as dictionaries with a
`text` and an `id` field, so each match retains a reference to its source
text via `match.text_id`.


In [4]:
texts = [
    {
        "text": ("ie Veucris den 5. Januaris 1725. PR&ASIDE, Den Heere Bentinck. PRASENTIEBUS, "
                  "De Heeren Jan Welderen , van Dam, Torck , met een extraordinaris Gedeputeerde "
                  "uyt de Provincie van Gelderlandt."),
        "id": "session-3780-num-3-para-1"
    },
    {
        "text": ("Mercuri: den 10. Jangarii, 1725. ia PRESIDE, Den Heere an Iddekinge. PRA&SENTIBUS, "
                  "De Heeren /an Welderen , van Dam, van Wynbergen, Torck."),
        "id": "session-3780-num-7-para-1"
    },
]

for text in texts:
    for match in fuzzy_searcher.find_matches(text):
        print(f"{match.phrase.phrase_string:<22}{match.string:<22}offset={match.offset:<4}"
              f"sim={match.levenshtein_similarity:.2f}\t{match.text_id}")


Veneris               Veucris               offset=3   sim=0.71	session-3780-num-3-para-1
den .. Januarii 1725  den 5. Januaris 1725  offset=11  sim=0.90	session-3780-num-3-para-1
PRAESIDE              PR&ASIDE              offset=33  sim=0.88	session-3780-num-3-para-1
PRAESENTIBUS          PRASENTIEBUS          offset=63  sim=0.92	session-3780-num-3-para-1
Mercurii              Mercuri               offset=0   sim=0.93	session-3780-num-7-para-1
den .. Januarii 1725  den 10. Jangarii, 1725offset=9   sim=0.86	session-3780-num-7-para-1
PRAESIDE              PRESIDE               offset=36  sim=0.93	session-3780-num-7-para-1
PRAESENTIBUS          PRA&SENTIBUS          offset=69  sim=0.92	session-3780-num-7-para-1


## 3. Phrase variants and distractors

`PRAESENTIBUS` ("those present") is sometimes written in plainer Dutch as
*Present de Heeren*. But a third, *visually similar but semantically
different*, formulaic phrase, *Praeside de Heer* (announcing the chair, not
the attendants), would also match `PRAESENTIBUS` by accident if we are not
careful.

We solve this by:
- registering `Present de Heeren` as a **variant** of `PRAESENTIBUS`
  (`include_variants: True` enables variant matching), and
- registering `Praeside de Heer` as a **distractor** of `PRAESENTIBUS`
  (`filter_distractors: True` removes matches that are closer to a
  distractor than to the phrase itself).


In [5]:
attendants_phrases = [
    {
        "phrase": "PRAESENTIBUS",
        "variants": ["Present de Heeren", "Pntes die voors"],
        "distractors": ["Praeside de Heer"],
        "label": "attendants",
    },
    {
        "phrase": "PRAESIDE",
        "variants": ["Praeside de Heer"],
        "label": "president",
    },
]

config = {
    "char_match_threshold": 0.6,
    "ngram_threshold": 0.5,
    "levenshtein_threshold": 0.6,
    "include_variants": True,
    "filter_distractors": True,
    "max_length_variance": 3,
    "ngram_size": 2,
    "skip_size": 2,
}

fuzzy_searcher = FuzzyPhraseSearcher(config=config, phrase_model=attendants_phrases)

text = {
    "text": "Praeside de Heer de Knuijt, Praseat de Heeron Verbolt, Raesfelt, Huijgens.",
    "id": "session-3210-num-166-para-1",
}

for match in fuzzy_searcher.find_matches(text):
    print(f"{match.phrase.phrase_string:<14}{match.string:<20}offset={match.offset:<4}"
          f"label={match.label_list}")


PRAESIDE      Praeside de Heer    offset=0   label=['president']
PRAESENTIBUS  Praseat de Heeron   offset=28  label=['attendants']


Both phrases are recognized correctly: *Praeside de Heer* is attributed to
`PRAESIDE`, and *Praseat de Heeron* (an OCR-damaged form of *Present de
Heeren*) is attributed to `PRAESENTIBUS` -- without `PRAESENTIBUS` falsely
matching the *Praeside de Heer* phrase.

## 4. Templates: requiring phrases to co-occur in order

Beyond single phrases, `fuzzy-search` can require several labeled phrases to
appear together, in a specific order, to recognize a higher-level pattern --
for example, an auction announcement naming a broker followed by goods for
sale. This also works well for the formulaic structure of resolutions, where
the chair is named before the attendants.


In [6]:
from fuzzy_search.phrase.phrase_model import PhraseModel
from fuzzy_search.pattern.fuzzy_template import FuzzyTemplate
from fuzzy_search.search.template_searcher import FuzzyTemplateSearcher

simple_phrases = [
    {"phrase": "PRAESIDE", "label": "president"},
    {"phrase": "PRAESENTIBUS", "label": "attendants"},
]
phrase_model = PhraseModel(phrases=simple_phrases)
template = ["president", "attendants"]
fuzzy_template = FuzzyTemplate(phrase_model, template)

template_searcher = FuzzyTemplateSearcher(fuzzy_template, {})

template_text = ("PR&ASIDE, Den Heere Bentinck. PRASENTIEBUS, "
                  "De Heeren Jan Welderen, van Dam, Torck.")

phrase_matches = template_searcher.find_matches(template_text)
template_matches = template_searcher.find_template_matches(phrase_matches)

for template_match in template_matches:
    for element_match in template_match.element_matches:
        print("Template element:", element_match["label"])
        for phrase_match in element_match["phrase_matches"]:
            print(f"\t{phrase_match.phrase.phrase_string:<14}{phrase_match.string:<20}"
                  f"offset={phrase_match.offset}")


Template element: president
	PRAESIDE      PR&ASIDE            offset=0
Template element: attendants
	PRAESENTIBUS  PRASENTIEBUS        offset=30


The template confirms that this resolution opening contains both a
`president` element and an `attendants` element, each linked back to the
specific phrase matches that satisfied them.

## Next steps

See the [modern English text notebook](modern_english_text.ipynb) for the
same approach applied to noisy auction advertisement text, and the
[readthedocs Quick Start](https://fuzzy-search.readthedocs.io/) for the full
configuration reference.
